In [1]:
import torch
from torch import nn
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

In [ ]:
## THIS CODE IS TO CREATE THE TENSOR FOR THE TRAINING DATA ##

jw_df = pd.read_csv('../../Data/Final/TT Split/jw_train_long.csv')

# Pivot to create a time series for each (x, y)
jw_pivot = jw_df.pivot(index=['x', 'y'], columns='Date', values='Value')
#print (jw_pivot)

# Reset index to keep (x, y) as columns
jw_pivot = jw_pivot.reset_index()

# Convert time columns back into a NumPy array
ts_jw = jw_pivot.iloc[:, 2:].values  # Ignore first two columns (x, y)
loc_jw = jw_pivot.iloc[:, :2].values  # Store coordinates

# Standardization with Z-score normalisation
jw_mean_LST = ts_jw.mean()
jw_std_LST = ts_jw.std()
ts_jw = (ts_jw - jw_mean_LST) / jw_std_LST 

# Convert to PyTorch tensor wiith extra dimension for LST
X_train_tensor_jw = torch.tensor(ts_jw, dtype=torch.float32).unsqueeze(-1)
#print(X_train_tensor.shape)

X_train_jw = X_train_tensor_jw
y_train_jw = X_train_tensor_jw[:, 1:, :]

#### Without weight decay and dropout

In [3]:
## THIS CODE IS TO DEFINE THE LSTM MODEL ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # LSTM output
        out = self.fc(lstm_out)  # Fully connected layer for final prediction
        return out

In [ ]:
## THIS CODE IS TO TRAIN THE MODEL ##
torch.manual_seed(5188)

model = LSTMPredictor() # Model
criterion = nn.MSELoss() # Loss function to track performance over epochs
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer (can be switched)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(X_train_jw)

    loss = criterion(predictions[:, :-1, :], y_train_jw)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0021
Epoch 10/100, Loss: 0.9807
Epoch 20/100, Loss: 0.9276
Epoch 30/100, Loss: 0.9141
Epoch 40/100, Loss: 0.9062
Epoch 50/100, Loss: 0.8995
Epoch 60/100, Loss: 0.8915
Epoch 70/100, Loss: 0.8793
Epoch 80/100, Loss: 0.8622
Epoch 90/100, Loss: 0.8454


In [ ]:
## THIS CODE IS FOR FORECASTING##

model.eval()
torch.manual_seed(5188)

with torch.no_grad():
    X_pred_jw = model(X_train_jw)  # Forecast next steps
    X_pred_jw= X_pred_jw[:, -12:, :]  # Extract last 12 bimonthly periods (2023-2024)

# Convert predictions to a NumPy array
X_pred_jw_np = X_pred_jw.squeeze().cpu().numpy()

# **Denormalize predictions**
X_pred_jw_np = (X_pred_jw_np * jw_std_LST) + jw_mean_LST  # Convert back to original from scaled values

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create a long-format DataFrame
jw_pred_df = pd.DataFrame({
    "x": np.repeat(loc_jw[:, 0], len(bimonthly_periods)),  # Use stored locations
    "y": np.repeat(loc_jw[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_jw),
    "Predicted_LST": X_pred_jw_np.flatten()  # Store denormalized values
})

jw_pred_df.to_csv('jw_pred_long.csv', index=False) # Save to CSV for RMSE

In [ ]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("jw_pred_long.csv")
pred_df = pd.read_csv("../../Data/Final/TT Split/jw_test_long.csv")

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Predicted_LST"] - merged_df["Value"]) ** 2))
print(f"RMSE between true and predicted LST for Jurong West is: {rmse:.4f}")

            x         y          Date  Predicted_LST      Value
0  103.710171  1.307789  Mar-Apr 2023      25.094046  24.773416
1  103.710171  1.307789  May-Jun 2023      24.714594  22.167969
2  103.710171  1.307789  Jul-Aug 2023      24.520730  23.192685
3  103.710171  1.307789  Sep-Oct 2023      24.594967  24.916249
4  103.710171  1.307789  Nov-Dec 2023      24.888432  20.413347
RMSE between true and predicted LST for Jurong East is: 2.8506


#### With weight decay and dropout

In [13]:
# Define LSTM model with Dropout
class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out)
        return out

In [ ]:
# Initialize model
torch.manual_seed(5188)
model = LSTMPredictor()

# Define loss function and optimizer with weight decay
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)  # Added weight decay

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_jw)
    loss = criterion(predictions[:, :-1, :], y_train_jw)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0021
Epoch 10/100, Loss: 0.9809
Epoch 20/100, Loss: 0.9270
Epoch 30/100, Loss: 0.9146
Epoch 40/100, Loss: 0.9068
Epoch 50/100, Loss: 0.9002
Epoch 60/100, Loss: 0.8927
Epoch 70/100, Loss: 0.8814
Epoch 80/100, Loss: 0.8654
Epoch 90/100, Loss: 0.8470


In [ ]:
# Forecasting
model.eval()
torch.manual_seed(5188)
with torch.no_grad():
    X_pred_jw = model(X_train_jw)
    X_pred_jw = X_pred_jw[:, -12:, :]

# Denormalize predictions
X_pred_jw_np = (X_pred_jw.squeeze().cpu().numpy() * jw_std_LST) + jw_mean_LST

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create DataFrame for predictions
jw_pred_df = pd.DataFrame({
    "x": np.repeat(loc_jw[:, 0], len(bimonthly_periods)),
    "y": np.repeat(loc_jw[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_jw),
    "Predicted_LST": X_pred_jw_np.flatten()
})

# Save predictions to CSV
jw_pred_df.to_csv('jw_pred_long.csv', index=False)


In [ ]:
# Compute RMSE
true_df = pd.read_csv("jw_pred_long.csv")
pred_df = pd.read_csv("../../Data/Final/TT Split/jw_test_long.csv")
merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
rmse = np.sqrt(np.mean((merged_df["Predicted_LST"] - merged_df["Value"]) ** 2))
print(f"RMSE between true and predicted LST for Jurong West is: {rmse:.4f}")

RMSE between true and predicted LST for JE is: 2.8442
